# 00 · Pilot: 1 epoch süre ölçümü (P0)

Amaç: 10k COCO alt kümesinde YOLOX-Nano'nun **1 eğitim epoch'unun** ve bir **val2017 değerlendirmesinin** gerçek süresini ve Colab kredi maliyetini ölçmek → NAS bütçe hesabı.

- Runtime: **L4 GPU** (Runtime → Change runtime type)
- Kod: `Bey3fendi/edge-ai-object-detection` + YOLOX fork `@3ec6263`
- Veri: 10k COCO train2017 alt kümesi ve val2017 Drive'da zip olarak saklanır, kullanımda `/content` yereline kopyalanır.
- Çıktılar: `results/pilot/` (Drive'da `edge_ai/results/pilot/`)
- Hücre 4'teki Drive bağlantısı tarayıcı onayı ister; yeni runtime'da bu hücreyi elle çalıştır.

Hücreler 1–5 GPU gerektirmez; eğitim adımına kadar CPU runtime yeterli.

> **Notlar (2026-10-04):** Hücre 6 kaynağı canlı log sürümüdür (subprocess ile satır akışı, eğitimde henüz çalıştırılmadı; aynı yöntem hücre 10'da çalıştı); kayıtlı çıktısı aynı ayarlarla önceki `!train.py | tail` sürümünden alınmıştır. Hücre 9 ikinci kez çalıştırıldığı için çıktısı `hazırlık 0.0 sn` gösterir; ilk ölçüm 20,0 sn'dir (env JSON'da kayıtlı). Hücre 10'un kayıtlı çıktısındaki tqdm satırları, filtre sonradan sadeleştirilmeden önceki çalıştırmadandır.

In [1]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

{
  "timestamp_utc": "2026-10-04T17:34:18+00:00",
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "gpu": "NVIDIA L4, 23034 MiB, 580.82.07",
  "cuda_nvcc": "Build cuda_13.0.r13.0/compiler.36424714_0",
  "cpu_count": 12,
  "ram_gb": 56.9,
  "disk_free_content": "overlay         236G   43G  194G  18% /",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "torch_cuda": "13.0",
  "cuda_available": true
}


In [2]:
# 2 · Kod: depo + YOLOX fork sabit commit
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)

{'repo_commit': 'a285b68', 'yolox_commit': '3ec6263', 'yolox_dirty': False}


In [3]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 12.3 MB/s eta 0:00:00
yolox import OK: /content/edge_ai_workspace/YOLOX/yolox/__init__.py
{'pkg_torch': '2.11.0+cu130', 'pkg_torchvision': '0.26.0+cu130', 'pkg_numpy': '2.1.3', 'pkg_opencv-python-headless': '5.0.0.93', 'pkg_pycocotools': '2.0.11', 'pkg_loguru': '0.7.3', 'pkg_thop': '0.1.1-2209072238'}


In [5]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/pilot"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

Mounted at /content/drive
cd28f55fbbc1829f99d9ac9b38a16d259a22889739c8728ea877610201feff7b 7694953


In [6]:
# 5 · 10k COCO train2017 alt kümesi (sınıf dağılımı korunmuş / stratified)
# Yöntem: her görsel, içerdiği en nadir sınıfa (train2017 örnek sayısına göre) atanır; bu "birincil
# sınıf" katmanlarından orantılı (largest remainder) örnek çekilir. Anotasyonsuz görseller hariç.
# Seed sabit değil (pilot kararı); seçilen ID listesi Drive'a kaydedildiği için alt küme yine
# yeniden üretilebilir. Drive'da zip varsa yeniden seçim yapılmaz, yalnız yerele kopyalanır.
import json, zipfile, random, collections, concurrent.futures as cf, time

N_SUBSET = 10_000
COCO_LOCAL = "/content/datasets/COCO"
SUBSET_ZIP = f"{DRIVE_DATA}/coco_train2017_10k_stratified.zip"
ANN_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip"
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)

# Anotasyonlar: Drive'da yoksa indir, yerele aç (val json'u da YOLOX evaluator kurulumu için gerekli)
if not os.path.exists(ANN_ZIP):
    urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", ANN_ZIP)
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
        if not os.path.exists(f"{COCO_LOCAL}/{name}"):
            z.extract(name, COCO_LOCAL)

t0 = time.time()
if not os.path.exists(SUBSET_ZIP):
    full = json.load(open(f"{COCO_LOCAL}/annotations/instances_train2017.json"))
    img_cats = collections.defaultdict(set)
    inst_count = collections.Counter()
    for a in full["annotations"]:
        img_cats[a["image_id"]].add(a["category_id"])
        inst_count[a["category_id"]] += 1
    # birincil sınıf = görseldeki en nadir sınıf
    strata = collections.defaultdict(list)
    for img_id, cats in img_cats.items():
        strata[min(cats, key=lambda c: inst_count[c])].append(img_id)
    total = sum(len(v) for v in strata.values())
    quota = {c: N_SUBSET * len(v) / total for c, v in strata.items()}
    take = {c: int(q) for c, q in quota.items()}
    for c in sorted(quota, key=lambda c: quota[c] - take[c], reverse=True)[: N_SUBSET - sum(take.values())]:
        take[c] += 1
    rng = random.Random()
    chosen = set()
    for c, ids in strata.items():
        chosen.update(rng.sample(ids, take[c]))
    assert len(chosen) == N_SUBSET

    sub = {k: full[k] for k in ("info", "licenses", "categories")}
    sub["images"] = [im for im in full["images"] if im["id"] in chosen]
    sub["annotations"] = [a for a in full["annotations"] if a["image_id"] in chosen]

    # Dağılım kontrolü: sınıf başına örnek payı (alt küme vs tam)
    sub_count = collections.Counter(a["category_id"] for a in sub["annotations"])
    n_full, n_sub = sum(inst_count.values()), sum(sub_count.values())
    dev = {c: abs(sub_count[c] / n_sub - inst_count[c] / n_full) * 100 for c in inst_count}
    print(f"alt küme: {len(sub['images'])} görsel, {n_sub} örnek; sınıf payı sapması "
          f"maks {max(dev.values()):.3f} puan, ort {sum(dev.values())/len(dev):.3f} puan; "
          f"eksik sınıf: {[c for c in inst_count if sub_count[c] == 0]}")
    del full

    # Görseller: yalnız seçilen 10k indirilir
    img_dir = f"{COCO_LOCAL}/train2017"
    os.makedirs(img_dir, exist_ok=True)
    def fetch(im):
        dst = f"{img_dir}/{im['file_name']}"
        if not os.path.exists(dst):
            urllib.request.urlretrieve(f"http://images.cocodataset.org/train2017/{im['file_name']}", dst)
    with cf.ThreadPoolExecutor(32) as ex:
        list(ex.map(fetch, sub["images"]))

    ann_path = f"{COCO_LOCAL}/annotations/instances_train2017_10k.json"
    json.dump(sub, open(ann_path, "w"))
    with open(f"{COCO_LOCAL}/subset_10k_image_ids.json", "w") as f:
        json.dump(sorted(chosen), f)
    # Drive'a tek zip (JPEG zaten sıkıştırılmış → STORED)
    tmp_zip = "/content/coco10k.zip"
    with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_STORED) as z:
        z.write(ann_path, "annotations/instances_train2017_10k.json")
        z.write(f"{COCO_LOCAL}/subset_10k_image_ids.json", "subset_10k_image_ids.json")
        for im in sub["images"]:
            z.write(f"{img_dir}/{im['file_name']}", f"train2017/{im['file_name']}")
    shutil.copy(tmp_zip, SUBSET_ZIP)
    env["subset_source"] = "created"
else:
    shutil.copy(SUBSET_ZIP, "/content/coco10k.zip")
    with zipfile.ZipFile("/content/coco10k.zip") as z:
        z.extractall(COCO_LOCAL)
    env["subset_source"] = "drive"

n_imgs = len(os.listdir(f"{COCO_LOCAL}/train2017"))
env["subset_images"] = n_imgs
env["subset_prep_s"] = round(time.time() - t0, 1)
print(f"yerel train2017: {n_imgs} görsel · hazırlık {env['subset_prep_s']} sn · kaynak {env['subset_source']}")
!du -sh {COCO_LOCAL}/train2017 {SUBSET_ZIP}

yerel train2017: 10000 görsel · hazırlık 43.4 sn · kaynak drive
1.6G	/content/datasets/COCO/train2017
1.6G	/content/drive/MyDrive/edge_ai/data/coco_train2017_10k_stratified.zip


In [10]:
# 6 · Pilot eğitim: 1 epoch, 10k alt küme, L4 (GPU gerekir)
# Dikkat (YOLOX trainer.py): no_aug = start_epoch >= max_epoch - no_aug_epochs (satır 150) ve
# before_epoch'ta epoch+1 == max_epoch - no_aug_epochs (satır 220) → mosaic kapanır, L1 açılır,
# eval_interval=1 olur (val2017 değerlendirmesi). max_epoch=1 için ikisini de atlatmak üzere
# no_aug_epochs=-1. Exp dosyası YOLOX submodule'ü DIŞINDA tutulur.
import torch, re, time, shutil, subprocess, datetime as dt
assert torch.cuda.is_available(), "GPU yok: Runtime → Change runtime type → L4"
env["gpu"] = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

BATCH = 64
WORKERS = os.cpu_count()
EXP_FILE = f"{WS}/exps/pilot_nano_10k.py"
os.makedirs(os.path.dirname(EXP_FILE), exist_ok=True)
with open(EXP_FILE, "w") as f:
    f.write(f'''from yolox.exp.default.yolox_nano import Exp as _Base


class Exp(_Base):
    def __init__(self):
        super().__init__()
        self.data_dir = "{COCO_LOCAL}"
        self.train_ann = "instances_train2017_10k.json"
        self.val_ann = "instances_val2017.json"
        self.max_epoch = 1
        self.no_aug_epochs = -1  # 0 bile mosaic'i kapatır (bkz. hücre başı)
        self.eval_interval = 10_000
        self.data_num_workers = {WORKERS}
        self.print_interval = 10
        self.save_history_ckpt = False
        self.exp_name = "pilot_nano_10k"
''')

OUT = f"{WS}/YOLOX_outputs/pilot_nano_10k"
shutil.rmtree(OUT, ignore_errors=True)

# Canlı log: yalnız ilerleme/uyarı/hata satırları akıtılır (model tablosu ve exp dökümü atlanır)
SHOW = re.compile(r"iter: \d+/|start train|Save weights|Training of experiment|mosaic|WARNING|ERROR|Error|Traceback")
t0 = time.time()
proc = subprocess.Popen(
    ["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH), "--fp16",
     "-c", WEIGHTS, "-expn", "pilot_nano_10k"],
    cwd=WS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tail = []
for line in proc.stdout:
    tail = (tail + [line])[-30:]
    if SHOW.search(line):
        m = re.search(r"epoch: .*?(?=, lr)|^.*?\| (.*)$", line.strip())
        print(f"[{time.time()-t0:6.0f}s] {m.group(0) if m else line.strip()}", flush=True)
proc.wait()
wall_s = time.time() - t0
if proc.returncode != 0:
    print("".join(tail))
    raise RuntimeError(f"train.py çıkış kodu {proc.returncode}")

TRAIN_LOG = f"{OUT}/train_log.txt"
log = open(TRAIN_LOG).read()
assert "No mosaic aug now" not in log, "mosaic kapandı — pilot temsilî değil"
assert "Exception in training" not in log, "eğitim hatayla bitti"
def ts(pattern, last=False):
    lines = [l for l in log.splitlines() if pattern in l]
    return dt.datetime.strptime(lines[-1 if last else 0][:23], "%Y-%m-%d %H:%M:%S.%f")
epoch_s = (ts("Save weights to", last=True) - ts("start train epoch1")).total_seconds()
iters = [float(x) for x in re.findall(r"iter_time: ([\d.]+)s", log)]
datas = [float(x) for x in re.findall(r"data_time: ([\d.]+)s", log)]
n_iter = int(re.findall(r"iter: \d+/(\d+)", log)[-1])
env.update(batch=BATCH, workers=WORKERS, iters_per_epoch=n_iter,
           wall_s_total=round(wall_s, 1),   # süreç başlatma + model/veri kurulumu dahil
           epoch_s=round(epoch_s, 1),        # yalnız epoch: log zaman damgalarından
           iter_time_mean_s=round(sum(iters) / len(iters), 4),
           data_time_mean_s=round(sum(datas) / len(datas), 4))
print({k: env[k] for k in ("gpu", "batch", "workers", "iters_per_epoch", "wall_s_total",
                           "epoch_s", "iter_time_mean_s", "data_time_mean_s")})

2026-10-04 17:58:22 | INFO     | yolox.core.trainer:270 - epoch: 1/1, iter: 70/157, gpu mem: 7666Mb, mem: 6.6Gb, iter_time: 0.392s, data_time: 0.002s, total_loss: 6.0, iou_loss: 2.2, l1_loss: 0.0, conf_loss: 2.6, cls_loss: 1.2, lr: 7.952e-05, size: 384, ETA: 0:02:32
2026-10-04 17:58:43 | INFO     | yolox.core.trainer:270 - epoch: 1/1, iter: 80/157, gpu mem: 7666Mb, mem: 7.0Gb, iter_time: 2.127s, data_time: 0.002s, total_loss: 5.7, iou_loss: 2.2, l1_loss: 0.0, conf_loss: 2.4, cls_loss: 1.1, lr: 1.039e-04, size: 352, ETA: 0:02:18
2026-10-04 17:58:47 | INFO     | yolox.core.trainer:270 - epoch: 1/1, iter: 90/157, gpu mem: 7666Mb, mem: 7.1Gb, iter_time: 0.439s, data_time: 0.002s, total_loss: 5.8, iou_loss: 2.3, l1_loss: 0.0, conf_loss: 2.5, cls_loss: 1.1, lr: 1.314e-04, size: 480, ETA: 0:01:50
2026-10-04 17:59:10 | INFO     | yolox.core.trainer:270 - epoch: 1/1, iter: 100/157, gpu mem: 7666Mb, mem: 7.2Gb, iter_time: 2.273s, data_time: 0.002s, total_loss: 5.4, iou_loss: 2.1, l1_loss: 0.0, c

In [14]:
# 7 · Analiz: yavaş pencereler ilk kez görülen giriş boyutunda mı?
# (train.py cudnn.benchmark=True → her yeni giriş şeklinde cuDNN algoritma araması yapılır)
# Log satırındaki iter_time = son 10 iterasyonun ortalaması; "size" o pencerenin boyutudur
# (after_iter önce log yazar, sonra random_resize çağırır).
# Pilot sonucu (2026-10-04): ilk kez görülen 8/8 pencere yavaş, tekrarlanan 7/7 pencere hızlı.
rows = re.findall(r"iter: (\d+)/\d+, .*?iter_time: ([\d.]+)s, .*?size: (\d+)", log)
seen, fast, slow = set(), [], []
for it, t, size in rows:
    t, size = float(t), int(size)
    first = size not in seen
    seen.add(size)
    (slow if first else fast).append(t)
    print(f"iter {int(it):>3}  boyut {size}  {t:.3f}s  {'İLK KEZ' if first else ''}")
steady = sum(fast) / len(fast)
first_cost = (sum(slow) / len(slow) - steady) * 10   # boyut başına ek süre (10 iter/pencere)
n_sizes = len(range(exp.input_size[0] // 32 - exp.multiscale_range, exp.input_size[0] // 32 + exp.multiscale_range + 1))
print(f"\ntekrarlanan boyut: {steady:.3f}s/iter ({len(fast)} pencere) → kararlı epoch ≈ {steady*n_iter:.0f} sn"
      f"\nilk kez görülen boyut: {sum(slow)/len(slow):.3f}s/iter ({len(slow)} pencere) → boyut başına ≈ {first_cost:.0f} sn ek"
      f"\nolası boyut sayısı: {n_sizes} → koşu başına tek seferlik ≈ {first_cost*n_sizes:.0f} sn")
env.update(steady_iter_s=round(steady, 4), steady_epoch_s_est=round(steady * n_iter, 1),
           cudnn_first_shape_cost_s=round(first_cost, 1), n_input_sizes=n_sizes)

iter  10  boyut 416  2.582s  İLK KEZ
iter  20  boyut 384  2.236s  İLK KEZ
iter  30  boyut 448  2.141s  İLK KEZ
iter  40  boyut 384  0.406s  
iter  50  boyut 608  2.339s  İLK KEZ
iter  60  boyut 480  2.166s  İLK KEZ
iter  70  boyut 384  0.392s  
iter  80  boyut 352  2.127s  İLK KEZ
iter  90  boyut 480  0.439s  
iter 100  boyut 576  2.273s  İLK KEZ
iter 110  boyut 320  2.175s  İLK KEZ
iter 120  boyut 576  0.523s  
iter 130  boyut 352  0.389s  
iter 140  boyut 576  0.538s  
iter 150  boyut 320  0.382s  

tekrarlanan boyut: 0.438s/iter (7 pencere) → kararlı epoch ≈ 69 sn
ilk kez görülen boyut: 2.255s/iter (8 pencere) → boyut başına ≈ 18 sn ek
olası boyut sayısı: 11 → koşu başına tek seferlik ≈ 200 sn


In [16]:
# 8 · Sonuçları kaydet → Drive edge_ai/results/pilot/ (depoya results/pilot/ olarak alınır)
# CU_PER_HOUR: Colab "Resources" panelindeki L4 tüketim hızı (Furkan, 2026-10-04: ~1.54 CU/saat).
CU_PER_HOUR = 1.54
run_id = "pilot_" + dt.datetime.now(dt.UTC).strftime("%Y%m%dT%H%M%SZ")
env["run_id"] = run_id
env["cu_per_hour"] = CU_PER_HOUR
env["cu_per_epoch_10k"] = round(env["epoch_s"] / 3600 * CU_PER_HOUR, 4)
env["cu_per_epoch_10k_steady"] = round(env["steady_epoch_s_est"] / 3600 * CU_PER_HOUR, 4)

with open(f"{DRIVE_RESULTS}/{run_id}_env.json", "w") as f:
    json.dump(env, f, indent=2, default=str)
shutil.copy(TRAIN_LOG, f"{DRIVE_RESULTS}/{run_id}_train_log.txt")
shutil.copy(EXP_FILE, f"{DRIVE_RESULTS}/{run_id}_exp.py")
shutil.copy(f"{COCO_LOCAL}/subset_10k_image_ids.json", f"{DRIVE_RESULTS}/subset_10k_image_ids.json")

import csv
row = {k: env.get(k) for k in ("run_id", "gpu", "torch", "yolox_commit", "weights_sha256",
                               "subset_images", "batch", "workers", "iters_per_epoch", "epoch_s",
                               "iter_time_mean_s", "data_time_mean_s", "wall_s_total",
                               "steady_iter_s", "steady_epoch_s_est", "cudnn_first_shape_cost_s",
                               "n_input_sizes", "cu_per_hour", "cu_per_epoch_10k",
                               "cu_per_epoch_10k_steady")}
csv_path = f"{DRIVE_RESULTS}/pilot_epoch_timing.csv"
new = not os.path.exists(csv_path)
with open(csv_path, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row))
    if new:
        w.writeheader()
    w.writerow(row)
print(json.dumps(env, indent=2, default=str))
!ls -la {DRIVE_RESULTS}

{
  "timestamp_utc": "2026-10-04T17:34:18+00:00",
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "gpu": "NVIDIA L4, 23034 MiB, 580.82.07",
  "cuda_nvcc": "Build cuda_13.0.r13.0/compiler.36424714_0",
  "cpu_count": 12,
  "ram_gb": 56.9,
  "disk_free_content": "overlay         236G   43G  194G  18% /",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "torch_cuda": "13.0",
  "cuda_available": true,
  "repo_commit": "a285b68",
  "yolox_commit": "3ec6263",
  "yolox_dirty": false,
  "pkg_torch": "2.11.0+cu130",
  "pkg_torchvision": "0.26.0+cu130",
  "pkg_numpy": "2.1.3",
  "pkg_opencv-python-headless": "5.0.0.93",
  "pkg_pycocotools": "2.0.11",
  "pkg_loguru": "0.7.3",
  "pkg_thop": "0.1.1-2209072238",
  "weights_url": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth",
  "weights_sha256": "cd28f55fbbc1829f99d9ac9b38a16d259a22889739c8728ea877610201feff7b",
  "weights_bytes": 7694953,
  "subset_source": "drive

In [18]:
# 9 · val2017 (5000 görsel): Drive'da zip yoksa indir, yerele aç
# NAS'ta her aday val2017'de değerlendirilir (Karar 2); P0 bütçesi için bu sürenin ölçülmesi gerekir.
for v in ("WS", "COCO_LOCAL", "DRIVE_DATA", "WEIGHTS"):
    assert v in globals(), f"{v} yok: runtime sıfırlanmış, önce hücre 1–5'i çalıştır"
VAL_ZIP = f"{DRIVE_DATA}/val2017.zip"
t0 = time.time()
if not os.path.exists(VAL_ZIP):
    urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", VAL_ZIP)
if not os.path.isdir(f"{COCO_LOCAL}/val2017"):
    shutil.copy(VAL_ZIP, "/content/val2017.zip")
    with zipfile.ZipFile("/content/val2017.zip") as z:
        z.extractall(COCO_LOCAL)
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
env["val_prep_s"] = round(time.time() - t0, 1)
print(f"val2017: {env['val_images']} görsel · hazırlık {env['val_prep_s']} sn")
assert env["val_images"] == 5000

val2017: 5000 görsel · hazırlık 0.0 sn


In [20]:
# 10 · val2017 değerlendirme süresi (eğitim içi değerlendirmeyle aynı ayar: fp32, batch 64, conf 0.001, 416)
# Önceden eğitilmiş Nano ağırlığı kullanılır → aynı zamanda bu ortamda baseline mAP (25,8) kontrolü.
# Pilot sonucu (2026-10-04): eval 87,6 sn (≈38 sn GPU çıkarım + ≈48 sn CPU COCOeval); mAP 25,8 / AP50 41,4.
EVAL_EXPN = "pilot_eval_val2017"
EVAL_OUT = f"{WS}/YOLOX_outputs/{EVAL_EXPN}"
shutil.rmtree(EVAL_OUT, ignore_errors=True)
SHOW_EVAL = re.compile(r"Average forward|IoU=0.50:0.95 \| area=   all \| maxDets=100|IoU=0.50      \| area=   all|ERROR|Error|Traceback|100%\|")
t0 = time.time()
proc = subprocess.Popen(
    ["python", "-u", "YOLOX/tools/eval.py", "-f", EXP_FILE, "-c", WEIGHTS, "-b", "64", "-d", "1",
     "--conf", "0.001", "-expn", EVAL_EXPN],
    cwd=WS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tail, out_all = [], []
for line in proc.stdout:
    out_all.append(line)
    tail = (tail + [line])[-30:]
    if SHOW_EVAL.search(line):
        print(f"[{time.time()-t0:6.0f}s] {line.strip()[-160:]}", flush=True)
proc.wait()
eval_wall_s = time.time() - t0
if proc.returncode != 0:
    print("".join(tail))
    raise RuntimeError(f"eval.py çıkış kodu {proc.returncode}")

txt = "".join(out_all)
ap = re.search(r"Average Precision  \(AP\) @\[ IoU=0.50:0.95 \| area=   all \| maxDets=100 \] = ([\d.]+)", txt)
ap50 = re.search(r"Average Precision  \(AP\) @\[ IoU=0.50      \| area=   all \| maxDets=100 \] = ([\d.]+)", txt)
fwd = re.search(r"Average forward time: ([\d.]+) ms, Average NMS time: ([\d.]+) ms, Average inference time: ([\d.]+) ms", txt)
VAL_LOG = f"{EVAL_OUT}/val_log.txt"
vlog = open(VAL_LOG).read().splitlines()
t_start = dt.datetime.strptime(next(l for l in vlog if "loading checkpoint" in l)[:23], "%Y-%m-%d %H:%M:%S.%f")
t_end = dt.datetime.strptime([l for l in vlog if l[:4].isdigit()][-1][:23], "%Y-%m-%d %H:%M:%S.%f")
env.update(
    eval_wall_s=round(eval_wall_s, 1),                       # süreç başlatma dahil
    eval_s=round((t_end - t_start).total_seconds(), 1),      # checkpoint yükleme → sonuç (çıkarım + COCOeval)
    eval_map5095=round(float(ap.group(1)) * 100, 1) if ap else None,
    eval_ap50=round(float(ap50.group(1)) * 100, 1) if ap50 else None,
    eval_forward_ms=float(fwd.group(1)) if fwd else None,
    eval_nms_ms=float(fwd.group(2)) if fwd else None,
)
print({k: env[k] for k in ("eval_wall_s", "eval_s", "eval_map5095", "eval_ap50", "eval_forward_ms", "eval_nms_ms")})

[    49s] 100%|##########| 79/79 [00:38<00:00,  2.04it/s]
[    97s] Average forward time: 1.79 ms, Average NMS time: 0.86 ms, Average inference time: 2.65 ms
[    97s] Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.258
[    97s] Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.414
[    97s] Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.270
[    97s] Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.082
[    97s] Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.275
[    97s] Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.409
[    97s] Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.242
[    97s] Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.384
[    97s] Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.420
[    97s] Average Recall     (AR) @[ IoU=

In [21]:
# 11 · Değerlendirme sonuçlarını kaydet → Drive edge_ai/results/pilot/
eval_id = "pilot_eval_" + dt.datetime.now(dt.UTC).strftime("%Y%m%dT%H%M%SZ")
shutil.copy(VAL_LOG, f"{DRIVE_RESULTS}/{eval_id}_val_log.txt")
row = {"run_id": eval_id, "gpu": env["gpu"], "torch": env["torch"], "yolox_commit": env["yolox_commit"],
       "weights_sha256": env["weights_sha256"], "val_images": env["val_images"], "batch": 64,
       "precision": "fp32", "input_size": 416, "conf": 0.001,
       "eval_s": env["eval_s"], "eval_wall_s": env["eval_wall_s"],
       "forward_ms_per_img": env["eval_forward_ms"], "nms_ms_per_img": env["eval_nms_ms"],
       "map5095": env["eval_map5095"], "ap50": env["eval_ap50"],
       "cu_per_eval": round(env["eval_s"] / 3600 * CU_PER_HOUR, 4)}
csv_path = f"{DRIVE_RESULTS}/pilot_eval_timing.csv"
new = not os.path.exists(csv_path)
with open(csv_path, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row))
    if new:
        w.writeheader()
    w.writerow(row)
with open(f"{DRIVE_RESULTS}/{eval_id}_env.json", "w") as f:
    json.dump(env, f, indent=2, default=str)
print(json.dumps(row, indent=2))
!ls -la {DRIVE_RESULTS}

{
  "run_id": "pilot_eval_20261004T185107Z",
  "gpu": "NVIDIA L4, 23034 MiB, 580.82.07",
  "torch": "2.11.0+cu130",
  "yolox_commit": "3ec6263",
  "weights_sha256": "cd28f55fbbc1829f99d9ac9b38a16d259a22889739c8728ea877610201feff7b",
  "val_images": 5000,
  "batch": 64,
  "precision": "fp32",
  "input_size": 416,
  "conf": 0.001,
  "eval_s": 87.6,
  "eval_wall_s": 99.9,
  "forward_ms_per_img": 1.79,
  "nms_ms_per_img": 0.86,
  "map5095": 25.8,
  "ap50": 41.4,
  "cu_per_eval": 0.0375
}
total 167
drwx------ 2 root root  4096 Oct  4 18:51 .
drwx------ 3 root root  4096 Oct  4 17:22 ..
-rw------- 1 root root  1461 Oct  4 18:01 pilot_20261004T180128Z_env.json
-rw------- 1 root root   570 Oct  4 18:01 pilot_20261004T180128Z_exp.py
-rw------- 1 root root 43105 Oct  4 18:01 pilot_20261004T180128Z_train_log.txt
-rw------- 1 root root   495 Oct  4 18:01 pilot_epoch_timing.csv
-rw------- 1 root root  1641 Oct  4 18:51 pilot_eval_20261004T185107Z_env.json
-rw------- 1 root root 34357 Oct  4 18:51 p